## **Importing Libraries**

In [1]:
import pandas as pd
from sqlalchemy import create_engine,text

## **Loading dataset**

In [2]:
engine = create_engine('sqlite:///mydatabase.db')
# creating DataFrame
df = pd.read_csv("/kaggle/input/datasets/amruthayenikonda/dirty-dataset-to-practice-data-cleaning/my_file (1).csv")
# DataFrame to database
df.to_sql(name='female_singer_concerts', con=engine,if_exists="replace",method="multi",index=False)

20

## **Creating Temporary table**

In [3]:
df.to_sql(name='female_singer_concerts_cleaned', con=engine,if_exists="replace",method="multi",index=False)

20

## **Displaying dataset**

In [4]:
pd.read_sql("SELECT * FROM female_singer_concerts_cleaned;",engine)

,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Ref.
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571",[1]
1,2,1,7[2],"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571",[3]
2,3,1[4],2[5],"$411,000,000","$560,622,615",Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,"$4,835,294",[6]
3,4,2[7],10[7],"$397,300,000","$454,751,555",Pink,Beautiful Trauma World Tour,2018–2019,156,"$2,546,795",[7]
4,5,2[4],None,"$345,675,146","$402,844,849",Taylor Swift,Reputation Stadium Tour,2018,53,"$6,522,173",[8]
5,6,2[4],10[9],"$305,158,363","$388,978,496",Madonna,The MDNA Tour,2012,88,"$3,467,709",[9]
6,7,2[10],None,"$280,000,000","$381,932,682",Celine Dion,Taking Chances World Tour,2008–2009,131,"$2,137,405",[11]
7,7,None,None,"$257,600,000","$257,600,000",Pink,Summer Carnival †,2023–2024,41,"$6,282,927",[12]
8,9,None,None,"$256,084,556","$312,258,401",Beyoncé,The Formation World Tour,2016,49,"$5,226,215",[13]
9,10,None,None,"$250,400,000","$309,141,878",Taylor Swift,The 1989 World Tour,2015,85,"$2,945,882",[14]


### **Counting number of records in table**

In [5]:
pd.read_sql("SELECT COUNT(*) FROM female_singer_concerts_cleaned;",engine)

,COUNT(*)
0,20


### **Finding Duplicates**

In [6]:
pd.read_sql("SELECT Artist,COUNT(*) FROM female_singer_concerts_cleaned GROUP BY Artist;",engine)

,Artist,COUNT(*)
0,Adele,1
1,Beyoncé,3
2,Celine Dion,1
3,Cher,1
4,Katy Perry,1
5,Lady Gaga,2
6,Madonna,4
7,Pink,3
8,Taylor Swift,4


In [7]:
query = """
SELECT Artist,
COUNT(*) OVER(PARTITION BY Artist,`Tour title`) 
FROM female_singer_concerts_cleaned;
"""
pd.read_sql(query,engine)

,Artist,"COUNT(*) OVER(PARTITION BY Artist,`Tour title`)"
0,Adele,1
1,Beyoncé,1
2,Beyoncé,1
3,Beyoncé,1
4,Celine Dion,1
5,Cher,1
6,Katy Perry,1
7,Lady Gaga,1
8,Lady Gaga,1
9,Madonna,1


#### **Checking duplicates in 'Tour title' column**

In [8]:
query = """
SELECT `Tour title`,
COUNT(`Tour title`) FROM female_singer_concerts_cleaned 
GROUP BY `Tour title` 
HAVING 
COUNT(`Tour title`)>1;"""
pd.read_sql(query,engine)

,Tour title,COUNT(`Tour title`)


##### Observation
- There are no duplicates in the Tour title column.

#### **Checking duplicates in 'Rank' column**

In [9]:
query = """
WITH RankDuplicateCTE AS 
(
SELECT `Rank`,
ROW_NUMBER() OVER(PARTITION BY `Rank`) AS rank_duplicate,
`Tour Title` FROM female_singer_concerts_cleaned)

SELECT * 
FROM RankDuplicateCTE 
WHERE rank_duplicate>1;
"""
pd.read_sql(query,engine)

,Rank,rank_duplicate,Tour Title
0,7,2,Summer Carnival †


##### Observation
- There is a duplicate row under the column 'Rank' with rank 7.

### **Datatype checking**

In [10]:
pd.read_sql("PRAGMA table_info(female_singer_concerts_cleaned);",engine)

,cid,name,type,notnull,dflt_value,pk
0,0,Rank,BIGINT,0,None,0
1,1,Peak,TEXT,0,None,0
2,2,All Time Peak,TEXT,0,None,0
3,3,Actual gross,TEXT,0,None,0
4,4,Adjusted gross (in 2022 dollars),TEXT,0,None,0
5,5,Artist,TEXT,0,None,0
6,6,Tour title,TEXT,0,None,0
7,7,Year(s),TEXT,0,None,0
8,8,Shows,BIGINT,0,None,0
9,9,Average gross,TEXT,0,None,0


## **Data Cleaning**

### **Dropping irrelevant column**

In [11]:
# Deleting the column 'Ref.' in the table.
with engine.connect() as connection:
    connection.execute(
        text(
            "ALTER TABLE female_singer_concerts_cleaned DROP COLUMN `Ref.`"
        )
    )
    connection.commit()

In [12]:
pd.read_sql("SELECT * FROM female_singer_concerts_cleaned LIMIT 5",engine)

,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571"
1,2,1,7[2],"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571"
2,3,1[4],2[5],"$411,000,000","$560,622,615",Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,"$4,835,294"
3,4,2[7],10[7],"$397,300,000","$454,751,555",Pink,Beautiful Trauma World Tour,2018–2019,156,"$2,546,795"
4,5,2[4],None,"$345,675,146","$402,844,849",Taylor Swift,Reputation Stadium Tour,2018,53,"$6,522,173"


### **Change column names**

In [13]:
pd.read_sql("SELECT `Rank` AS `rank_of_artist` FROM female_singer_concerts_cleaned;",engine)

,rank_of_artist
0,1
1,2
2,3
3,4
4,5
5,6
6,7
7,7
8,9
9,10


In [14]:
with engine.connect() as connection:
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN `Rank` TO `rank_of_artist`")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN `All Time Peak` TO all_time_peak;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN `Peak` TO `peak`;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN `Shows` TO `number_of_shows`;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN Artist TO artist;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN `Tour title` TO `tour_title`;")
    )
    connection.commit()

In [15]:
pd.read_sql("SELECT * FROM female_singer_concerts_cleaned LIMIT 5",engine)

,rank_of_artist,peak,all_time_peak,Actual gross,Adjusted gross (in 2022 dollars),artist,tour_title,Year(s),number_of_shows,Average gross
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571"
1,2,1,7[2],"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571"
2,3,1[4],2[5],"$411,000,000","$560,622,615",Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,"$4,835,294"
3,4,2[7],10[7],"$397,300,000","$454,751,555",Pink,Beautiful Trauma World Tour,2018–2019,156,"$2,546,795"
4,5,2[4],None,"$345,675,146","$402,844,849",Taylor Swift,Reputation Stadium Tour,2018,53,"$6,522,173"


In [16]:
# Renaming columns in the table.
with engine.connect() as connection:
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN `Actual gross` TO actual_gross;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN `Adjusted gross (in 2022 dollars)` TO `adjusted_gross_2022_usd`;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN `Year(s)` to year;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN `Average gross` to average_gross;")
    )
    connection.commit()

In [17]:
# Display the table.
pd.read_sql("SELECT * FROM female_singer_concerts_cleaned LIMIT 5;",engine)

,rank_of_artist,peak,all_time_peak,actual_gross,adjusted_gross_2022_usd,artist,tour_title,year,number_of_shows,average_gross
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571"
1,2,1,7[2],"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571"
2,3,1[4],2[5],"$411,000,000","$560,622,615",Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,"$4,835,294"
3,4,2[7],10[7],"$397,300,000","$454,751,555",Pink,Beautiful Trauma World Tour,2018–2019,156,"$2,546,795"
4,5,2[4],None,"$345,675,146","$402,844,849",Taylor Swift,Reputation Stadium Tour,2018,53,"$6,522,173"


#### **Removing Irregularities in data**

In [18]:
with engine.connect() as connection:
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET peak=TRIM(peak)")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET all_time_peak=TRIM(all_time_peak);")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET actual_gross=TRIM(actual_gross);")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET adjusted_gross_2022_usd=TRIM(adjusted_gross_2022_usd);")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET artist=TRIM(artist);")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET tour_title=TRIM(tour_title);")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET year=TRIM(year);")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET average_gross=TRIM(average_gross);")
    )
    connection.commit()

In [19]:
query = """
SELECT peak,
CASE 
WHEN INSTR(peak,'[')>=1 THEN SUBSTR(peak,0,INSTR(peak,'[')) 
ELSE peak 
END as changed,
SUBSTR(peak,0,INSTR(peak,'[')) 
FROM female_singer_concerts_cleaned;
"""
pd.read_sql(query,engine)

,peak,changed,"SUBSTR(peak,0,INSTR(peak,'['))"
0,1,1,
1,1,1,
2,1[4],1,1
3,2[7],2,2
4,2[4],2,2
5,2[4],2,2
6,2[10],2,2
7,None,None,None
8,None,None,None
9,None,None,None


In [20]:
with engine.connect() as connection:
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET peak=CASE WHEN INSTR(peak,'[')>=1 THEN SUBSTR(peak,0,INSTR(peak,'[')) ELSE peak END;")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET all_time_peak=CASE WHEN INSTR(all_time_peak,'[')>=1 THEN SUBSTR(all_time_peak,0,INSTR(all_time_peak,'[')) ELSE all_time_peak END;")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET actual_gross=CASE WHEN INSTR(actual_gross,'[')>=1 THEN SUBSTR(actual_gross,0,INSTR(actual_gross,'[')) ELSE actual_gross END;")
    )
    connection.commit()

In [21]:
pd.read_sql("SELECT * FROM female_singer_concerts_cleaned LIMIT 10;",engine)

,rank_of_artist,peak,all_time_peak,actual_gross,adjusted_gross_2022_usd,artist,tour_title,year,number_of_shows,average_gross
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571"
1,2,1,7,"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571"
2,3,1,2,"$411,000,000","$560,622,615",Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,"$4,835,294"
3,4,2,10,"$397,300,000","$454,751,555",Pink,Beautiful Trauma World Tour,2018–2019,156,"$2,546,795"
4,5,2,None,"$345,675,146","$402,844,849",Taylor Swift,Reputation Stadium Tour,2018,53,"$6,522,173"
5,6,2,10,"$305,158,363","$388,978,496",Madonna,The MDNA Tour,2012,88,"$3,467,709"
6,7,2,None,"$280,000,000","$381,932,682",Celine Dion,Taking Chances World Tour,2008–2009,131,"$2,137,405"
7,7,None,None,"$257,600,000","$257,600,000",Pink,Summer Carnival †,2023–2024,41,"$6,282,927"
8,9,None,None,"$256,084,556","$312,258,401",Beyoncé,The Formation World Tour,2016,49,"$5,226,215"
9,10,None,None,"$250,400,000","$309,141,878",Taylor Swift,The 1989 World Tour,2015,85,"$2,945,882"


In [22]:
query = """
SELECT tour_title,
CASE 
WHEN INSTR(tour_title,'[')>=1 THEN SUBSTR(tour_title,0,INSTR(tour_title,'[')) 
ELSE tour_title 
END FROM female_singer_concerts_cleaned
"""
pd.read_sql(query,engine)

,tour_title,"CASE \nWHEN INSTR(tour_title,'[')>=1 THEN SUBSTR(tour_title,0,INSTR(tour_title,'[')) \nELSE tour_title \nEND"
0,The Eras Tour †,The Eras Tour †
1,Renaissance World Tour,Renaissance World Tour
2,Sticky & Sweet Tour ‡[4][a],Sticky & Sweet Tour ‡
3,Beautiful Trauma World Tour,Beautiful Trauma World Tour
4,Reputation Stadium Tour,Reputation Stadium Tour
5,The MDNA Tour,The MDNA Tour
6,Taking Chances World Tour,Taking Chances World Tour
7,Summer Carnival †,Summer Carnival †
8,The Formation World Tour,The Formation World Tour
9,The 1989 World Tour,The 1989 World Tour


In [23]:
with engine.connect() as connection:
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET actual_gross=REPLACE(REPLACE(actual_gross,'$',''),',','');")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET adjusted_gross_2022_usd=REPLACE(REPLACE(adjusted_gross_2022_usd,'$',''),',','');")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET average_gross=REPLACE(REPLACE(average_gross,'$',''),',','');")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET tour_title=CASE WHEN INSTR(tour_title,'[')>=1 THEN SUBSTR(tour_title,0,INSTR(tour_title,'[')) ELSE tour_title END;")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET tour_title=TRIM(REPLACE(REPLACE(REPLACE(tour_title,'†',''),'‡',''),'*',''));")
    )
    connection.commit()

In [24]:
pd.read_sql("SELECT * FROM female_singer_concerts_cleaned LIMIT 10;",engine)

,rank_of_artist,peak,all_time_peak,actual_gross,adjusted_gross_2022_usd,artist,tour_title,year,number_of_shows,average_gross
0,1,1,2,780000000,780000000,Taylor Swift,The Eras Tour,2023–2024,56,13928571
1,2,1,7,579800000,579800000,Beyoncé,Renaissance World Tour,2023,56,10353571
2,3,1,2,411000000,560622615,Madonna,Sticky & Sweet Tour,2008–2009,85,4835294
3,4,2,10,397300000,454751555,Pink,Beautiful Trauma World Tour,2018–2019,156,2546795
4,5,2,None,345675146,402844849,Taylor Swift,Reputation Stadium Tour,2018,53,6522173
5,6,2,10,305158363,388978496,Madonna,The MDNA Tour,2012,88,3467709
6,7,2,None,280000000,381932682,Celine Dion,Taking Chances World Tour,2008–2009,131,2137405
7,7,None,None,257600000,257600000,Pink,Summer Carnival,2023–2024,41,6282927
8,9,None,None,256084556,312258401,Beyoncé,The Formation World Tour,2016,49,5226215
9,10,None,None,250400000,309141878,Taylor Swift,The 1989 World Tour,2015,85,2945882


In [25]:
with engine.connect() as connection:
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET rank_of_artist=8 WHERE tour_title LIKE 'Summer Carnival';")
    )
    connection.commit()

In [26]:
query="""
WITH DuplicateRankCTE AS 
(
SELECT rank_of_artist,
ROW_NUMBER() 
OVER(PARTITION BY rank_of_artist) AS rank_duplicate 
FROM female_singer_concerts_cleaned)

SELECT * FROM DuplicateRankCTE 
WHERE rank_duplicate>1;
"""
pd.read_sql(query,engine)

,rank_of_artist,rank_duplicate


##### Observation:
- There are no duplicate rows with rank. Duplicate rank is fixed.

### **Changing Datatype of columns**

In [27]:
with engine.connect() as connection:
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned ADD all_time_peak_2 INT;")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET all_time_peak_2=CAST(all_time_peak AS UNSIGNED) WHERE all_time_peak NOT NULL;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned DROP COLUMN all_time_peak;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN all_time_peak_2 TO all_time_peak;")
    )
    connection.commit()

In [28]:
with engine.connect() as connection:
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned ADD peak_copy INT;")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET peak_copy=CAST(peak AS UNSIGNED) WHERE peak NOT NULL;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned DROP COLUMN peak;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN peak_copy TO peak;")
    )
    connection.commit()

In [29]:
with engine.connect() as connection:
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned ADD actual_gross_usd_copy INT;")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET actual_gross_usd_copy=CAST(actual_gross AS UNSIGNED) WHERE actual_gross NOT NULL;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned DROP COLUMN actual_gross;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN actual_gross_usd_copy TO actual_gross_usd;")
    )
    connection.commit()

In [30]:
with engine.connect() as connection:
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned ADD adjusted_gross_2022_usd_copy INT;")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET adjusted_gross_2022_usd_copy=CAST(adjusted_gross_2022_usd AS UNSIGNED) WHERE adjusted_gross_2022_usd NOT NULL;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned DROP COLUMN adjusted_gross_2022_usd;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN adjusted_gross_2022_usd_copy TO adjusted_gross_2022_usd;")
    )
    connection.commit()

In [31]:
with engine.connect() as connection:
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned ADD average_gross_usd_copy INT;")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET average_gross_usd_copy=CAST(average_gross AS UNSIGNED) WHERE average_gross NOT NULL;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned DROP COLUMN average_gross;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned RENAME COLUMN average_gross_usd_copy TO average_gross_usd;")
    )
    connection.commit()

In [32]:
pd.read_sql("PRAGMA table_info(female_singer_concerts_cleaned);",engine)

,cid,name,type,notnull,dflt_value,pk
0,0,rank_of_artist,BIGINT,0,None,0
1,1,artist,TEXT,0,None,0
2,2,tour_title,TEXT,0,None,0
3,3,year,TEXT,0,None,0
4,4,number_of_shows,BIGINT,0,None,0
5,5,all_time_peak,INT,0,None,0
6,6,peak,INT,0,None,0
7,7,actual_gross_usd,INT,0,None,0
8,8,adjusted_gross_2022_usd,INT,0,None,0
9,9,average_gross_usd,INT,0,None,0


### **Feature Engineering**

In [33]:
with engine.connect() as connection:
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned ADD start_year INT;")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned ADD end_year INT;")
    )
    connection.commit()

In [34]:
with engine.connect() as connection:
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET start_year=CAST(SUBSTR(year,0,5) AS UNSIGNED);")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET end_year=CAST(SUBSTR(year,-4,5) AS UNSIGNED);")
    )
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned DROP COLUMN year;")
    )
    connection.commit()

In [35]:
with engine.connect() as connection:
    connection.execute(
        text("ALTER TABLE female_singer_concerts_cleaned ADD concert_duration_in_years INT;")
    )
    connection.execute(
        text("UPDATE female_singer_concerts_cleaned SET concert_duration_in_years=end_year-start_year;")
    )
    connection.commit()

In [36]:
pd.read_sql("SELECT * FROM female_singer_concerts_cleaned LIMIT 5;",engine)

,rank_of_artist,artist,tour_title,number_of_shows,all_time_peak,peak,actual_gross_usd,adjusted_gross_2022_usd,average_gross_usd,start_year,end_year,concert_duration_in_years
0,1,Taylor Swift,The Eras Tour,56,2.0,1,780000000,780000000,13928571,2023,2024,1
1,2,Beyoncé,Renaissance World Tour,56,7.0,1,579800000,579800000,10353571,2023,2023,0
2,3,Madonna,Sticky & Sweet Tour,85,2.0,1,411000000,560622615,4835294,2008,2009,1
3,4,Pink,Beautiful Trauma World Tour,156,10.0,2,397300000,454751555,2546795,2018,2019,1
4,5,Taylor Swift,Reputation Stadium Tour,53,NaN,2,345675146,402844849,6522173,2018,2018,0


### **Data validation after cleaning**

In [37]:
pd.read_sql("SELECT COUNT(*) FROM female_singer_concerts_cleaned;",engine)

,COUNT(*)
0,20


In [38]:
pd.read_sql("PRAGMA table_info(female_singer_concerts_cleaned);",engine)

,cid,name,type,notnull,dflt_value,pk
0,0,rank_of_artist,BIGINT,0,None,0
1,1,artist,TEXT,0,None,0
2,2,tour_title,TEXT,0,None,0
3,3,number_of_shows,BIGINT,0,None,0
4,4,all_time_peak,INT,0,None,0
5,5,peak,INT,0,None,0
6,6,actual_gross_usd,INT,0,None,0
7,7,adjusted_gross_2022_usd,INT,0,None,0
8,8,average_gross_usd,INT,0,None,0
9,9,start_year,INT,0,None,0


### **Cleaned dataset**

In [39]:
pd.read_sql("SELECT * FROM female_singer_concerts_cleaned;",engine)

,rank_of_artist,artist,tour_title,number_of_shows,all_time_peak,peak,actual_gross_usd,adjusted_gross_2022_usd,average_gross_usd,start_year,end_year,concert_duration_in_years
0,1,Taylor Swift,The Eras Tour,56,2.0,1.0,780000000,780000000,13928571,2023,2024,1
1,2,Beyoncé,Renaissance World Tour,56,7.0,1.0,579800000,579800000,10353571,2023,2023,0
2,3,Madonna,Sticky & Sweet Tour,85,2.0,1.0,411000000,560622615,4835294,2008,2009,1
3,4,Pink,Beautiful Trauma World Tour,156,10.0,2.0,397300000,454751555,2546795,2018,2019,1
4,5,Taylor Swift,Reputation Stadium Tour,53,NaN,2.0,345675146,402844849,6522173,2018,2018,0
5,6,Madonna,The MDNA Tour,88,10.0,2.0,305158363,388978496,3467709,2012,2012,0
6,7,Celine Dion,Taking Chances World Tour,131,NaN,2.0,280000000,381932682,2137405,2008,2009,1
7,8,Pink,Summer Carnival,41,NaN,NaN,257600000,257600000,6282927,2023,2024,1
8,9,Beyoncé,The Formation World Tour,49,NaN,NaN,256084556,312258401,5226215,2016,2016,0
9,10,Taylor Swift,The 1989 World Tour,85,NaN,NaN,250400000,309141878,2945882,2015,2015,0


## Saving cleaned dataset as CSV file

In [40]:
cleaned_dataset = pd.read_sql("SELECT * FROM female_singer_concerts_cleaned;",engine)
cleaned_dataset.to_csv("cleaned_dataset.csv")

## **Analysis on cleaned dataset**

### What are the top 5 concert tours with highest average gross amount earned?

In [41]:
query = """
SELECT tour_title,
average_gross_usd,
artist,
rank_of_artist
FROM female_singer_concerts_cleaned
ORDER BY average_gross_usd
DESC
LIMIT 5;
"""
pd.read_sql(query,engine)

,tour_title,average_gross_usd,artist,rank_of_artist
0,The Eras Tour,13928571,Taylor Swift,1
1,Renaissance World Tour,10353571,Beyoncé,2
2,Reputation Stadium Tour,6522173,Taylor Swift,5
3,Summer Carnival,6282927,Pink,8
4,The Formation World Tour,5226215,Beyoncé,9


### What is the total actual grossing amount made by each artist?

In [42]:
query = """
SELECT artist,
SUM(actual_gross_usd) as total_actual_gross_in_usd
FROM female_singer_concerts_cleaned
GROUP BY artist
ORDER BY SUM(actual_gross_usd)
DESC;
"""
pd.read_sql(query,engine)

,artist,total_actual_gross_in_usd
0,Taylor Swift,1526075146
1,Madonna,1079958363
2,Beyoncé,1064984556
3,Pink,838900000
4,Lady Gaga,397400000
5,Celine Dion,280000000
6,Katy Perry,204000000
7,Cher,200000000
8,Adele,167700000


### Find the highest grossing concert tours in each year which ended within the last 5 years?

In [43]:
query = """
WITH HighGrossingConcertByYearCTE AS
(SELECT *,
DENSE_RANK() OVER(PARTITION BY end_year ORDER BY actual_gross_usd DESC) AS high_gross_rank
FROM female_singer_concerts_cleaned)

SELECT end_year,
tour_title,
artist,
actual_gross_usd,
high_gross_rank
FROM HighGrossingConcertByYearCTE
WHERE high_gross_rank=1
ORDER BY end_year DESC
LIMIT 5;
"""
pd.read_sql(query,engine)

,end_year,tour_title,artist,actual_gross_usd,high_gross_rank
0,2024,The Eras Tour,Taylor Swift,780000000,1
1,2023,Renaissance World Tour,Beyoncé,579800000,1
2,2019,Beautiful Trauma World Tour,Pink,397300000,1
3,2018,Reputation Stadium Tour,Taylor Swift,345675146,1
4,2017,Adele Live 2016,Adele,167700000,1


### Find the top 3 artists with the highest number of shows before 2016?

In [44]:
query = """
SELECT artist,
SUM(number_of_shows) AS total_number_of_shows_before_2016
FROM female_singer_concerts_cleaned
WHERE start_year<2016
GROUP BY artist
ORDER BY SUM(number_of_shows)
DESC
LIMIT 3;
"""
pd.read_sql(query,engine)

,artist,total_number_of_shows_before_2016
0,Cher,325
1,Madonna,315
2,Lady Gaga,301


### Find the top 3 artists with the highest number of shows from 2016?

In [45]:
query = """
SELECT artist,
SUM(number_of_shows) AS total_number_of_shows_from_2016
FROM female_singer_concerts_cleaned
WHERE start_year>2016
GROUP BY artist
ORDER BY SUM(number_of_shows)
DESC;
"""
pd.read_sql(query,engine)

,artist,total_number_of_shows_from_2016
0,Pink,197
1,Taylor Swift,109
2,Beyoncé,56


### Find the concert tour with highest number of shows?

In [46]:
query="""SELECT artist as Artist,
tour_title as `Tour title`,
number_of_shows
FROM female_singer_concerts_cleaned
ORDER BY number_of_shows
DESC
LIMIT 1;"""
pd.read_sql(query,engine)

,Artist,Tour title,number_of_shows
0,Cher,Living Proof: The Farewell Tour,325


### What is the overall average gross amount made by each artist?

In [47]:
query = """
SELECT artist,
AVG(actual_gross_usd) AS `Average gross amount in USD`
FROM female_singer_concerts_cleaned 
GROUP BY artist 
ORDER BY artist;
"""
pd.read_sql(query,engine)

,artist,Average gross amount in USD
0,Adele,1.677000e+08
1,Beyoncé,3.549949e+08
2,Celine Dion,2.800000e+08
3,Cher,2.000000e+08
4,Katy Perry,2.040000e+08
5,Lady Gaga,1.987000e+08
6,Madonna,2.699896e+08
7,Pink,2.796333e+08
8,Taylor Swift,3.815188e+08
